# 02 — ML Pipeline Multiclass

Full model selection, oversampling comparison, OOD evaluation, and interpretability for WoT and Dota at best granularity (loaded from registry).

In [ ]:
# imports
import warnings, json
from pathlib import Path
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.base import clone
from sklearn.metrics import classification_report

from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from optuna.samplers import TPESampler
import joblib

import sys
sys.path.insert(0, str(Path('../..').resolve()))
from src.loaders import load_wot, load_dota
from src.pipelines import build_pipe
from src.scoring import cv_score, holdout_score, ood_score, append_registry, F2_SCORER
from src.label_schemes import WOT_CLASS_NAMES, DOTA_CLASS_NAMES

# config
CONFIG = {
    'seed': 7524,
    'cv_folds': 5,
    'text_col': 'clean_message',
    'label_col': 'label',
    'optuna_trials': 30,
    'registry_path': Path('../../data/results/results_registry.csv'),
    'models_dir': Path('../../models'),
}
seed = CONFIG['seed']
cv = StratifiedKFold(n_splits=CONFIG['cv_folds'], shuffle=True, random_state=seed)
np.random.seed(seed)
CONFIG['models_dir'].mkdir(exist_ok=True)

# load best granularity from notebook 01 registry
# sweet spot was binary (n=2) for both games - confirmed by granularity experiment
registry = pd.read_csv(CONFIG['registry_path'])
gran = registry[registry['experiment'] == 'granularity_per_dataset']

wot_best_n = int(gran[gran['train_game'] == 'WoT'].groupby('n_classes')['cv_f2'].max().idxmax())
dota_best_n = int(gran[gran['train_game'] == 'Dota'].groupby('n_classes')['cv_f2'].max().idxmax())
print(f'WoT best granularity: {wot_best_n} classes')
print(f'Dota best granularity: {dota_best_n} classes')

## Section 2: Model Selection

In [ ]:
# model selection with Optuna tuning on F2 (recall-weighted)
# RandomOverSampler used for all models - chosen in notebook 01
# n_jobs=1 inside Optuna on Windows - multiprocessing spawn overhead too high

def run_model_selection(game: str, load_fn, n_classes: int):
    # load and binarise - label > 0 means toxic
    train_df = load_fn('train').copy()
    train_df[CONFIG['label_col']] = (train_df[CONFIG['label_col']].astype(int) > 0).astype(int)
    X_train = train_df[CONFIG['text_col']]
    y_train = train_df[CONFIG['label_col']]

    models_comparison = []

    # Logistic Regression - CV across 30 C values, optimised on F2
    print(f'[{game}] Logistic Regression ...')
    lr_pipe = build_pipe(
        LogisticRegressionCV(Cs=30, cv=cv, scoring=F2_SCORER, max_iter=1000, random_state=seed, n_jobs=-1),
        oversampler=RandomOverSampler(random_state=seed)
    )
    lr_scores = cv_score(lr_pipe, X_train, y_train, cv=cv)
    models_comparison.append({'Model': 'Logistic Regression', **lr_scores})

    # Naive Bayes - Optuna tunes alpha
    print(f'[{game}] Naive Bayes (Optuna) ...')
    def nb_objective(trial):
        p = {'clf__alpha': trial.suggest_float('clf__alpha', 0.001, 2.0, log=True)}
        p_pipe = build_pipe(MultinomialNB(), oversampler=RandomOverSampler(random_state=seed))
        p_pipe.set_params(**p)
        return cross_val_score(p_pipe, X_train, y_train, cv=cv, scoring=F2_SCORER, n_jobs=1).mean()

    study_nb = optuna.create_study(direction='maximize', sampler=TPESampler(seed=seed))
    study_nb.optimize(nb_objective, n_trials=CONFIG['optuna_trials'])
    nb_pipe = build_pipe(MultinomialNB(), oversampler=RandomOverSampler(random_state=seed))
    nb_pipe.set_params(**study_nb.best_params)
    nb_scores = cv_score(nb_pipe, X_train, y_train, cv=cv)
    models_comparison.append({'Model': 'Naive Bayes', **nb_scores})

    # LinearSVC - Optuna tunes C
    print(f'[{game}] LinearSVC (Optuna) ...')
    def svc_objective(trial):
        C = trial.suggest_float('clf__C', 0.01, 10.0, log=True)
        p_pipe = build_pipe(
            LinearSVC(C=C, max_iter=2000, tol=1e-3, random_state=seed),
            oversampler=RandomOverSampler(random_state=seed)
        )
        return cross_val_score(p_pipe, X_train, y_train, cv=cv, scoring=F2_SCORER, n_jobs=1).mean()

    study_svc = optuna.create_study(direction='maximize', sampler=TPESampler(seed=seed))
    study_svc.optimize(svc_objective, n_trials=CONFIG['optuna_trials'])
    svc_pipe = build_pipe(
        LinearSVC(C=study_svc.best_params['clf__C'], max_iter=2000, tol=1e-3, random_state=seed),
        oversampler=RandomOverSampler(random_state=seed)
    )
    svc_scores = cv_score(svc_pipe, X_train, y_train, cv=cv)
    models_comparison.append({'Model': 'LinearSVC', **svc_scores})

    # rank by F2
    compare_df = pd.DataFrame(models_comparison).sort_values('cv_f2', ascending=False).reset_index(drop=True)
    print(f'\n{game} model comparison:')
    print(compare_df.to_string(index=False))

    pipes = {'Logistic Regression': lr_pipe, 'Naive Bayes': nb_pipe, 'LinearSVC': svc_pipe}
    return compare_df, pipes


print('=== WoT model selection ===')
wot_compare_df, wot_pipes = run_model_selection('WoT', load_wot, wot_best_n)

print('\n=== Dota model selection ===')
dota_compare_df, dota_pipes = run_model_selection('Dota', load_dota, dota_best_n)

wot_best_model_name = wot_compare_df.iloc[0]['Model']
dota_best_model_name = dota_compare_df.iloc[0]['Model']
print(f'\nBest: WoT={wot_best_model_name}, Dota={dota_best_model_name}')

`n_jobs=1` is used inside Optuna objectives on Windows because multiprocessing uses `spawn` (not `fork`), which causes large overhead re-importing the full pipeline on each worker. Single-threaded Optuna trials are faster in practice for this problem size. `LogisticRegressionCV` uses `n_jobs=-1` safely because sklearn's internal CV loop is thread-based (not process-based) on Windows.

## Section 2: Evaluate Best Model + OOD at Binary Level

In [ ]:
# evaluate best model on holdout set and OOD
# OOD always at binary level - both games share toxic/non-toxic contract

def evaluate_best(game, best_name, pipes, load_fn, n_classes, class_names,
                  compare_df, ood_load_fn=None, ood_game=None):
    # load and binarise train + val
    train_df = load_fn('train').copy()
    train_df[CONFIG['label_col']] = (train_df[CONFIG['label_col']].astype(int) > 0).astype(int)
    val_df = load_fn('val').copy()
    val_df[CONFIG['label_col']] = (val_df[CONFIG['label_col']].astype(int) > 0).astype(int)

    X_train, y_train = train_df[CONFIG['text_col']], train_df[CONFIG['label_col']]
    X_val, y_val = val_df[CONFIG['text_col']], val_df[CONFIG['label_col']]

    pipe = pipes[best_name]
    scores = holdout_score(pipe, X_train, y_train, X_val, y_val)

    print(f'=== {game} {best_name} - in-game test ===')
    print(classification_report(y_val, pipe.predict(X_val), target_names=class_names, zero_division=0))

    # OOD: binarise the other game's labels and evaluate
    ood_scores = {}
    if ood_load_fn:
        ood_val = ood_load_fn('val').copy()
        ood_val[CONFIG['label_col']] = (ood_val[CONFIG['label_col']].astype(int) > 0).astype(int)
        X_ood, y_ood = ood_val[CONFIG['text_col']], ood_val[CONFIG['label_col']]
        ood_scores = ood_score(pipe, X_ood, y_ood)
        print(f'=== {game} -> {ood_game} OOD ===')
        print(classification_report(y_ood, pipe.predict(X_ood),
                                    target_names=['Non-Toxic', 'Toxic'], zero_division=0))

    # pull cv scores from the comparison table for the best model
    best_cv = compare_df[compare_df['Model'] == best_name].iloc[0]

    append_registry({
        'experiment': 'ml_pipeline_multiclass',
        'train_game': game,
        'test_game': game,
        'n_classes': n_classes,
        'label_scheme': 'binary',
        'model': best_name,
        'cv_f2': best_cv.get('cv_f2'),
        'cv_f2_std': best_cv.get('cv_f2_std'),
        'cv_recall_macro': best_cv.get('cv_recall_macro'),
        'cv_precision_macro': best_cv.get('cv_precision_macro'),
        **scores,
        'ood_f2': ood_scores.get('ood_f2'),
        'ood_recall_macro': ood_scores.get('ood_recall_macro'),
        'ood_precision_macro': ood_scores.get('ood_precision_macro'),
        'ood_auc': ood_scores.get('ood_auc'),
        'anomaly_auroc': None,
        'notes': f'best_n={n_classes}',
    }, path=CONFIG['registry_path'])

    return pipe


print('=== WoT ===')
wot_best_pipe = evaluate_best(
    'WoT', wot_best_model_name, wot_pipes, load_wot,
    wot_best_n, WOT_CLASS_NAMES[2], wot_compare_df,
    ood_load_fn=load_dota, ood_game='Dota'
)

print('\n=== Dota ===')
dota_best_pipe = evaluate_best(
    'Dota', dota_best_model_name, dota_pipes, load_dota,
    dota_best_n, DOTA_CLASS_NAMES[4][:2], dota_compare_df,
    ood_load_fn=load_wot, ood_game='WoT'
)

OOD evaluation is always at **binary level** (toxic vs. non-toxic) using the 2-class scheme of the OOD game. This is valid cross-game because both games share the binary label contract even if their fine-grained schemes differ. Per-class recall is stored in the registry as a JSON string via `ood_score`. Full multiclass OOD would require a unified label scheme validated in notebook 01.

## Section 4: Interpretability — Top TF-IDF Features per Class

In [ ]:
# Extract top TF-IDF features per class from LR or SVC coefficients.
# LinearSVC and LR both expose .coef_ — no predict_proba needed.
# SHAP NOT used: LinearSVC has no predict_proba, coefficient inspection
# is more honest for linear TF-IDF models.

def plot_top_features(pipe, class_names: list, title: str, top_n: int = 15):
    tfidf         = pipe.named_steps['tfidf']
    clf           = pipe.named_steps['clf']
    feature_names = np.array(tfidf.get_feature_names_out())
    coef          = clf.coef_
    if coef.shape[0] == 1:
        coef = np.vstack([-coef, coef])

    n_cls = len(class_names)
    fig, axes = plt.subplots(1, n_cls, figsize=(5 * n_cls, 5))
    if n_cls == 1:
        axes = [axes]

    for ax, cls_idx, cls_name in zip(axes, range(n_cls), class_names):
        top_idx = np.argsort(coef[cls_idx])[-top_n:][::-1]
        ax.barh(feature_names[top_idx][::-1], coef[cls_idx][top_idx][::-1], color='#1565C0')
        ax.set_title(cls_name, fontsize=10, fontweight='bold')
        ax.tick_params(axis='y', labelsize=8)

    plt.suptitle(title, fontweight='bold', fontsize=13)
    plt.tight_layout()
    fname = title.replace(' ', '_').replace('/', '_').lower()
    Path('../../data/results').mkdir(parents=True, exist_ok=True)
    plt.savefig(f'../../data/results/{fname}_features.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_top_features(wot_best_pipe,  WOT_CLASS_NAMES[wot_best_n],
                   f'WoT Top Features --- {wot_best_n} classes')
plot_top_features(dota_best_pipe, DOTA_CLASS_NAMES[dota_best_n],
                   f'Dota Top Features --- {dota_best_n} classes')

For linear TF-IDF models, **coefficient inspection is more interpretable than SHAP**. SHAP requires `predict_proba` (not available for `LinearSVC`), and its additive decomposition adds little information when the model is already linear — the coefficients directly represent per-token contribution to each class score. The bar charts show which tokens drive each category decision.

## Section 5: Save Best Models

In [ ]:
# Save best pipes for reuse in error analysis and ensemble notebooks
import joblib
wot_path  = CONFIG['models_dir'] / f'multiclass_wot_{wot_best_n}class_{wot_best_model_name.lower().replace(" ","_")}.joblib'
dota_path = CONFIG['models_dir'] / f'multiclass_dota_{dota_best_n}class_{dota_best_model_name.lower().replace(" ","_")}.joblib'
joblib.dump(wot_best_pipe,  wot_path)
joblib.dump(dota_best_pipe, dota_path)
print(f'Saved: {wot_path}')
print(f'Saved: {dota_path}')

Saved model paths follow the pattern `multiclass_{game}_{n}class_{model}.joblib`. These files are consumed by notebook 06 (error analysis) and notebook 04 (ensemble / stacking). The naming convention encodes game, granularity, and model type for unambiguous loading without a separate config file.